# Do-Calculus

In the previous notebook, we saw that a DAG together with its factorisation allows us to compute a wide range of observational quantities using adjustment. By conditioning on an appropriate set of variables, we can block backdoor paths and isolate a causal effect.

However, this approach has a fundamental limitation. Not all causal effects are identifiable through simple covariate adjustment. In certain structures, particularly those involving colliders or more subtle forms of confounding, there may be no valid adjustment set at all, or naively adjusting can even introduce bias rather than remove it.

This raises a central question in causal inference:

> When does adjustment fail, and how do we proceed when it does?

To answer this, we move beyond backdoor adjustment and introduce the ideas of identification and do-calculus, which provide a more general framework for expressing causal quantities even in complex graphical structures.


**Conditioning vs Intervening**

We briefly recall the distinction introduced earlier:

- $P(Y \mid X=x)$ describes what we expect to observe when $X=x$ occurs naturally.  
- $P(Y \mid do(X=x))$ describes what would happen if we were to *set* $X=x$ by intervention.

These quantities coincide only under specific conditions (e.g. absence of confounding). In general, conditioning preserves the existing data-generating process, whereas intervention modifies it by severing the causal mechanisms that determine $X$.

The difference between these two perspectives is precisely what makes causal inference non-trivial: adjustment may recover the interventional quantity in some cases, but not in all.

>**Consider the following simple causal system**:
>
> A confounder $Z$ influences both treatment $X$ and outcome $Y$. 
>
> ![description](../figures/DAG2.png)
>
> We will compare the observational quantity $P(Y \mid X=1)$ with the interventional quantity $P(Y \mid do(X=1))$ via simulation.

In [ ]:
import numpy as np
import pandas as pd

N = 100_000

# Confounder
Z = np.random.binomial(1, 0.5, N)
# Treatment depends on Z
X = np.random.binomial(1, 0.2 + 0.6*Z)
# Outcome depends on BOTH X and Z
Y = np.random.binomial(1, 0.1 + 0.5*X + 0.3*Z)

df = pd.DataFrame({"Z": Z, "X": X, "Y": Y})

# Conditional probability P(Y=1|X=1)
cond_prob = df[(df["X"] == 1)]['Y'].mean()
print(f"P(Y=1 | X=1)     = {cond_prob:.3f}")

# Causal probability P(Y=1|do(X=1)
def simulate_do_x(x_value, n=100_000):
    Z = np.random.binomial(1, 0.5, n)
    X = np.full(n, x_value)  # force X
    Y = np.random.binomial(1, 0.1 + 0.5 * X + 0.3 * Z)
    return Y.mean()

int_prob = simulate_do_x(1)

print(f"P(Y=1 | do(X=1)) = {int_prob:.3f}")

P(Y=1 | X=1)     = 0.841
P(Y=1 | do(X=1)) = 0.751


> **Randomised Controlled Trials (RCTs)**
>
> In practice, the most reliable way to estimate causal effects is through randomised controlled trials (RCTs). Randomisation breaks all backdoor paths by construction, ensuring that treatment assignment is independent of all confounders. In such settings, simple comparisons of outcomes are already causal. In other words, randomisation renders the causal effect directly identifiable from the observed data, without the need for further adjustment or modelling assumptions.


In [2]:
# Randomise treatment assignment: X is generated independently of Z,
# mimicking an RCT where the confounder no longer influences treatment
p = 0.2
X = np.random.binomial(1, p, len(Z))
Y = np.random.binomial(1, 0.1 + 0.5*X + 0.3*Z)

df = pd.DataFrame({"Z": Z, "X": X, "Y": Y})


# Conditional probability P(Y=1|X=1)
cond_prob = df[(df["X"] == 1)]['Y'].mean()
print(f"P(Y=1 | X=1)     = {cond_prob:.3f}")
print(f"P(Y=1 | do(X=1)) = {int_prob:.3f}")

P(Y=1 | X=1)     = 0.755
P(Y=1 | do(X=1)) = 0.751


> As you can see above, randomisation removes systematic differences between treatment groups, allowing observed outcome differences to be interpreted causally rather than associationally.

However, RCTs are not always feasible. They may be:
- unethical (e.g. assigning harmful exposures),
- impractical (e.g. long-term societal effects),
- or simply impossible (e.g. historical events).

As a result, we turn to observational data, where causal structure must be inferred rather than enforced.

This leads to the central problem of **identification**:

> Can a causal effect be expressed purely in terms of observable quantities?

---

**Paths, d-Separation, and Blocking**

> To reason about causality in a DAG, we must understand how information flows along **paths**.
>
> A **path** is any sequence of connected nodes, regardless of edge direction. Along a path, three basic structures can occur:
>
> ![description](../figures/DAG3.png)

A path can be either **open** or **blocked**, depending on whether we condition on certain variables:

- Conditioning on a variable in a **chain** or **fork** blocks the path, *i.e.* once $Z$ is known, $X$ provides no additional information about $Y$ along that path  
- Conditioning on a **collider** (or its descendants) opens the path, *i.e.* conditioning induces a dependence between $X$ and $Y$ that was not present marginally  

This leads to the notion of **d-separation**:

> Two variables are d-separated by a set $Z$ if all paths between them are blocked when conditioning on $Z$.

D-separation provides a graphical criterion for conditional independence, and forms the foundation for identifying valid adjustment sets.

In [3]:
from utils import show_assoc

N = 100_000

# a) Chain: X -> Z -> Y  (X affects Y only through Z)
X_chain = np.random.binomial(1, 0.5, N)
Z_chain = np.random.binomial(1, 0.2 + 0.6*X_chain, N)
Y_chain = np.random.binomial(1, 0.1 + 0.5*Z_chain, N)
show_assoc(X_chain, Y_chain, Z_chain, "Chain (X -> Z -> Y)")

# b) Fork: X <- Z -> Y
Z_fork = np.random.binomial(1, 0.5, N)
X_fork = np.random.binomial(1, 0.2 + 0.6*Z_fork, N)
Y_fork = np.random.binomial(1, 0.1 + 0.5*Z_fork, N)
show_assoc(X_fork, Y_fork, Z_fork, "Fork (X <- Z -> Y)")

# c) Collider: X -> Z <- Y
X_coll = np.random.binomial(1, 0.5, N)
Y_coll = np.random.binomial(1, 0.5, N)
Z_coll = np.random.binomial(1, 0.05 + 0.45*X_coll + 0.45*Y_coll, N)
show_assoc(X_coll, Y_coll, Z_coll, "Collider (X -> Z <- Y)")

--------- Chain (X -> Z -> Y) ---------
P(Y=1 | X=1)      = 0.497    P(Y=1 | X=0) = 0.204
P(Y=1)            = 0.350

P(Y=1 | X=1, Z=0) = 0.104    P(Y=1 | X=0, Z=0) = 0.105
P(Y=1 | Z=0)      = 0.105

P(Y=1 | X=1, Z=1) = 0.596    P(Y=1 | X=0, Z=1) = 0.600
P(Y=1 | Z=1)      = 0.597


--------- Fork (X <- Z -> Y) ---------
P(Y=1 | X=1)      = 0.496    P(Y=1 | X=0) = 0.200
P(Y=1)            = 0.348

P(Y=1 | X=1, Z=0) = 0.101    P(Y=1 | X=0, Z=0) = 0.100
P(Y=1 | Z=0)      = 0.100

P(Y=1 | X=1, Z=1) = 0.597    P(Y=1 | X=0, Z=1) = 0.599
P(Y=1 | Z=1)      = 0.597


--------- Collider (X -> Z <- Y) ---------
P(Y=1 | X=1)      = 0.500    P(Y=1 | X=0) = 0.498
P(Y=1)            = 0.499

P(Y=1 | X=1, Z=0) = 0.091    P(Y=1 | X=0, Z=0) = 0.340
P(Y=1 | Z=0)      = 0.272

P(Y=1 | X=1, Z=1) = 0.655    P(Y=1 | X=0, Z=1) = 0.911
P(Y=1 | Z=1)      = 0.725




As above, we can verify these properties numerically:

**Chain & Fork — conditioning *blocks* the path:**

| | $P(Y=1 \mid X=1)$ | $P(Y=1 \mid X=0)$ |
|---|---|---|
| Marginal | $\approx 0.50$ | $\approx 0.20$ |
| Conditioning on $Z=0$ | $\approx 0.10$ | $\approx 0.10$ |
| Conditioning on $Z=1$ | $\approx 0.60$ | $\approx 0.60$ |

Marginally $X$ and $Y$ appear dependent, but once we condition on $Z$, the gap closes entirely — $P(Y=1 \mid X=1, Z=z) \approx P(Y=1 \mid X=0, Z=z)$ for both $z$, confirming $X \perp\!\!\!\perp Y \mid Z$.

**Collider — conditioning *opens* the path:**

| | $P(Y=1 \mid X=1)$ | $P(Y=1 \mid X=0)$ |
|---|---|---|
| Marginal | $\approx 0.50$ | $\approx 0.50$ |
| Conditioning on $Z=0$ | $\approx 0.09$ | $\approx 0.35$ |
| Conditioning on $Z=1$ | $\approx 0.65$ | $\approx 0.91$ |

Marginally $X$ and $Y$ are independent, but conditioning on $Z$ induces a strong spurious dependence — the gap *widens* rather than closes. This is the explaining away effect (Berkson's paradox): learning $Z$ makes $X$ and $Y$ mutually informative even though they share no direct causal link.

---

**Ignorability, Backdoor Criterion, and Adjustment**

In observational settings, causal effects can sometimes be recovered under the assumption of **ignorability**:
$
Y \perp\!\!\!\perp X \mid Z
$
meaning that, conditional on a set of variables $Z$, the treatment assignment behaves as if random.

In a graphical setting, this condition corresponds to blocking all non-causal paths between $X$ and $Y$. These are formalised as **backdoor paths**, defined as any path from $X$ to $Y$ that begins with an arrow into $X$, representing spurious associations due to confounding.

A set of variables $Z$ is called a **sufficient adjustment set** if it blocks all backdoor paths from $X$ to $Y$ (i.e. $Z$ d-separates $X$ and $Y$ when considering non-causal paths). Among such sets, a **minimal adjustment set** is one where no proper subset of $Z$ still satisfies this property.

Under these conditions, the causal effect becomes identifiable via the **adjustment formula**:
$$
P(Y \mid do(X)) = \sum_Z P(Y \mid X, Z)\,P(Z)
$$

Thus, ignorability and the backdoor criterion provide equivalent perspectives on when causal effects can be computed from observational data by appropriately conditioning on a set of variables.

> The causal effect is now estimated using the backdoor adjustment formula applied to observational data, rather than directly simulating interventions using the do-operator.

In [ ]:
# Backdoor adjustment formula to estimate causal effect of X on Y
# by controlling for confounder Z

# Empirical distribution of Z (P(Z))
p_z = df["Z"].value_counts(normalize=True)

adjusted = 0

# Apply backdoor adjustment:
for z in [0, 1]:
    
    # Estimate conditional outcome within stratum Z = z
    p_y_given_xz = df[(df["X"] == 1) & (df["Z"] == z)]["Y"].mean()
    
    # Weight by prevalence of that stratum
    adjusted += p_y_given_xz * p_z[z]

print(f"P(Y=1 | do(X=1)) via adjustment      = {adjusted:.3f}")
print(f"P(Y=1 | do(X=1) via RCT simulation)  = {cond_prob:.3f}")
print(f"P(Y=1 | do(X=1) via orignal process) = {int_prob:.3f}")

P(Y=1 | do(X=1)) via adjustment      = 0.754
P(Y=1 | do(X=1) via RCT simulation)  = 0.755
P(Y=1 | do(X=1) via orignal process) = 0.751


**When Adjustment Goes Wrong: Collider Bias and M-Bias**

While the backdoor criterion provides a principled way to select adjustment sets, it is crucial to recognise that not all conditioning is beneficial. In fact, conditioning on the wrong variables can introduce bias rather than remove it.

A key source of such bias is the **collider** structure:

> ![description](../figures/DAG4.png)

Here, $Z$ is a collider on the path between $X$ and $Y$. By default, this path is **blocked**. However, conditioning on $Z$ (or its descendants) **opens** the path, inducing a spurious association between $X$ and $Y$ even if none existed marginally. This is known as **collider bias**.

A more subtle example is **M-bias**, which arises in structures such as:

> ![description](../figures/DAG5.png)

In this case:
- The path between $X$ and $Y$ is initially blocked by the collider $Z$
- Conditioning on $Z$ opens the path, creating a non-causal association

Thus, adjusting for $Z$ introduces bias rather than removing it.

Valid adjustment requires carefully selecting variables that block backdoor paths **without opening new ones**. This reinforces the importance of graphical reasoning when applying the backdoor criterion.

In [10]:
N = 100_000

# 1. Collider bias example
# X -> Z <- Y
X = np.random.normal(size=N)
Y = np.random.normal(size=N)

# collider: Z depends on both X and Y
Z = X + Y + np.random.normal(scale=0.5, size=N)

df1 = pd.DataFrame({"X": X, "Y": Y, "Z": Z})

# marginal relationship
print("Collider example:")
print("Corr(X, Y) marginal:", df1["X"].corr(df1["Y"]))

# conditioning on Z induces dependence
subset = df1[np.abs(df1["Z"] - df1["Z"].mean()) < 0.1]
print("Corr(X, Y | Z ~ mean):", subset["X"].corr(subset["Y"]))


# 2. M-bias example
# X <- U1 -> Z <- U2 -> Y
U1 = np.random.normal(size=N)
U2 = np.random.normal(size=N)

# X and Y depends on U1 and U2 respectively
X = U1 + np.random.normal(scale=0.5, size=N)
Y = U2 + np.random.normal(scale=0.5, size=N)

# collider structure at Z
Z = U1 + U2 + np.random.normal(scale=0.5, size=N)

df2 = pd.DataFrame({"X": X, "Y": Y, "Z": Z})

# marginal relationship
print("\nM-bias example:")
print("Corr(X, Y) marginal:", df2["X"].corr(df2["Y"]))

# conditioning on Z induces dependence
subset2 = df2[np.abs(df2["Z"] - df2["Z"].mean()) < 0.1]
print("Corr(X, Y | Z ~ mean):", subset2["X"].corr(subset2["Y"]))

Collider example:
Corr(X, Y) marginal: -0.0044209808336359195
Corr(X, Y | Z ~ mean): -0.793392472297548

M-bias example:
Corr(X, Y) marginal: 0.007263749785496563
Corr(X, Y | Z ~ mean): -0.5388583905754087


**When No Adjustment Set Exists**

The backdoor criterion provides a powerful method for identifying causal effects, but it does not always apply.

In some graphs, there exists no set of observed variables that blocks all backdoor paths between $X$ and $Y$. This typically arises in the presence of **unobserved confounding**, where hidden variables influence both treatment and outcome.

In such cases:
- No sufficient adjustment set $Z$ exists  
- The ignorability condition fails  
- The causal effect $P(Y \mid do(X))$ cannot be recovered through adjustment alone  

This reveals a fundamental limitation:

> Not all causal effects are identifiable using covariate adjustment.

**Front-Door Criterion: Identification Beyond Backdoor Adjustment**

When no valid backdoor adjustment set exists, typically because common causes of $X$ and $Y$ are unobserved, causal effects may still be identifiable through alternative graph structures. The **front-door criterion** is one such case.

The criterion applies when there exists a mediator $Z$ satisfying:

1. $Z$ intercepts all directed paths from $X$ to $Y$
2. There are no unblocked backdoor paths from $X$ to $Z$
3. All backdoor paths from $Z$ to $Y$ are blocked by conditioning on $X$

> A typical structure would appear as:
>
> ![description](../figures/DAG6.png)
>
> Where U is an unobserved variable.

The intuition is a two-step argument. First, since condition 2 holds, the effect of $X$ on $Z$ is identifiable by simple conditioning: $P(Z \mid do(X)) = P(Z \mid X)$. Second, since condition 3 holds, the effect of $Z$ on $Y$ is identifiable by backdoor adjustment on $X$. Chaining these two steps together yields the front-door formula:

$$
\begin{aligned}
P(Y \mid do(X))
&= \sum_Z \underbrace{P(Z \mid do(X))}_{=P(Z \mid X)}
\underbrace{P(Y \mid Z, do(X))}_{\text{backdoor adjustment}} \\
&= \sum_Z P(Z \mid X)\sum_{X'} P(Y \mid Z, X') P(X')
\end{aligned}
$$



The key insight is that neither step requires observing the confounders between $X$ and $Y$. The mediator $Z$ acts as a bridge that makes the end-to-end effect recoverable purely from observational data.


In [11]:
N = 100_000

# Front-door example with hidden confounding U between X and Y
U = np.random.binomial(1, 0.5, N)
X = np.random.binomial(1, 0.2 + 0.5*U, N)   # X <- U -> Y
Z = np.random.binomial(1, 0.1 + 0.6*X, N)   # X -> Z -> Y
Y = np.random.binomial(1, 0.05 + 0.5*Z + 0.3*U, N)

df_front = pd.DataFrame({"U": U, "X": X, "Z": Z, "Y": Y})

# True causal quantity via intervention simulation
def simulate_do_x_front(x_value, n=100_000):
    U = np.random.binomial(1, 0.5, n)
    X = np.full(n, x_value)
    Z = np.random.binomial(1, 0.1 + 0.6*X)
    Y = np.random.binomial(1, 0.05 + 0.5*Z + 0.3*U)
    return Y.mean()

true_do1 = simulate_do_x_front(1)

print("Observed P(Y=1 | X=1):", df_front[df_front["X"] == 1]["Y"].mean())
print("True P(Y=1 | do(X=1)):", true_do1)

# Backdoor adjustment using U (if U were observed)
p_u = df_front["U"].value_counts(normalize=True).sort_index()
backdoor_with_u = sum(
    df_front[(df_front["X"] == 1) & (df_front["U"] == u)]["Y"].mean()*p_u.loc[u]
    for u in p_u.index)

print("\nBackdoor adjustment if U were observed:")
print(backdoor_with_u)
print()

# Front-door adjustment using only X and mediator Z
p_z_given_x1 = df_front[df_front["X"] == 1]["Z"].value_counts(normalize=True).sort_index()
p_y_given_zx = df_front.groupby(["Z", "X"])["Y"].mean()
p_x = df_front["X"].value_counts(normalize=True).sort_index()

frontdoor = 0.0
for z_val in [0, 1]:
    p_z_x1 = p_z_given_x1.get(z_val, 0.0)
    inner = sum(p_y_given_zx.loc[(z_val, x_val)] * p_x.loc[x_val] for x_val in [0, 1])
    frontdoor += p_z_x1 * inner

print("Front-door adjustment using Z if U were hidden:")
print(frontdoor)

Observed P(Y=1 | X=1): 0.6343218733985428
True P(Y=1 | do(X=1)): 0.55025

Backdoor adjustment if U were observed:
0.5497985589513944

Front-door adjustment using Z if U were hidden:
0.54830209901192


**Do-Calculus Framework**

The backdoor and frontdoor criteria provide powerful tools for identifying causal effects, but they apply only to specific classes of graphs. In more complex settings, we require a systematic way to determine whether a causal quantity can be expressed in terms of observational data.

**Do-calculus** provides such a framework.

Rather than relying on predefined criteria, do-calculus consists of a set of transformation rules that allow us to manipulate expressions involving interventions (i.e. $do(\cdot)$) into expressions involving only observational probabilities—when such a transformation is valid.

At a high level, do-calculus operates by:
- modifying the graph (e.g. removing incoming edges under intervention),
- applying d-separation in these modified graphs,
- and using the resulting independencies to justify algebraic transformations.

This turns causal inference into a structured process:

> Starting from a causal query such as $P(Y \mid do(X))$, repeatedly apply valid transformations until the expression depends only on observable quantities.
>
>![description](../figures/do_calculus_rules.png)

**Rule 1** reaffirms d-separation as a valid test for conditional independence in the distribution.

**Rule 2** provides a condition for an external intervention $do(Z = z)$ to have the same effect on Y as
the passive observation $Z = z$.

**Rule 3** provides conditions for introducing or deleting an external intervention $do(Z = z)$ without
affecting the probability of $Y = y$.


These rules are sound and complete for identification: if a causal effect can be expressed in terms of observational data, do-calculus provides a method to derive it. In this sense, backdoor and frontdoor criteria can be viewed as special cases that emerge naturally from the application of these more general rules.